# Fine-tuning y explicabilidad de BETO para noticias fake/no fake en español

Este notebook reúne en celdas ejecutables los módulos originales del proyecto:

1. configuración de rutas e hiperparámetros,
2. carga, limpieza y división estratificada del dataset,
3. fine-tuning de BETO,
4. evaluación en test,
5. explicabilidad con LIME, SHAP, Integrated Gradients y métricas de faithfulness.

> **Nota:** las celdas de entrenamiento/evaluación/explicabilidad están controladas por flags (`RUN_TRAIN`, `RUN_EVALUATE`, `RUN_EXPLAIN`) para evitar ejecutar procesos pesados por accidente.

## 1. Instalación de dependencias

Ejecuta esta celda solo si tu entorno no tiene ya instaladas las librerías necesarias.
En Colab normalmente conviene ejecutarla una vez y reiniciar el runtime si se actualizan paquetes importantes.

In [ ]:
# Descomenta si necesitas instalar dependencias en un entorno nuevo.
# %pip install -q pandas numpy scikit-learn datasets transformers accelerate torch lime shap captum safetensors

## 2. Imports base y comprobación de hardware

In [ ]:
from pathlib import Path
import json
import inspect
import re
import unicodedata
from html import unescape

import numpy as np
import pandas as pd
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configuración

Ajusta `DATASET_PATH` y `OUTPUT_DIR` según tu entorno. La ruta por defecto replica la de tus scripts originales.

En esta sección se separan dos tipos de parámetros:

- **Rutas y flags**, que controlan qué partes del flujo se ejecutan.
- **Dataclasses de configuración**, que agrupan hiperparámetros de entrenamiento y proporciones del dataset.


In [ ]:
# Rutas principales del proyecto.
# DATASET_PATH debe apuntar al CSV original de noticias.
# OUTPUT_DIR será la carpeta raíz donde se guardarán modelo, métricas y explicaciones.
DATASET_PATH = Path("../Data/Bin/Data_Bin_Classifier.csv")
OUTPUT_DIR = Path("./artifacts")
MODEL_DIR = OUTPUT_DIR / "model"

# Flags de ejecución.
# Están en False para evitar entrenar/evaluar/generar explicaciones por accidente,
# ya que esas tareas pueden tardar bastante según el tamaño del dataset y la GPU disponible.
RUN_TRAIN = False
RUN_EVALUATE = False
RUN_EXPLAIN = False

# Opciones de explicabilidad.
# Si EXPLAIN_TEXT contiene un texto, se explicará ese texto concreto.
# Si vale None, se usarán ejemplos del split de test.
EXPLAIN_TEXT = None  # ejemplo: "Texto de una noticia a explicar..."
EXPLAIN_N_SAMPLES = 3

# sample: calcula explicaciones para las primeras N muestras de test.
# full_test: calcula métricas de faithfulness sobre todo el test o sobre un muestreo limitado.
FAITHFULNESS_SCOPE = "sample"  # "sample" o "full_test"
FAITHFULNESS_MAX_SAMPLES = None


In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class TrainConfig:
    """Hiperparámetros principales para fine-tuning de BETO.

    La idea de usar una dataclass es tener todos los parámetros de entrenamiento
    en un único objeto fácil de modificar y reutilizar.
    """

    # Modelo base BETO de Hugging Face.
    model_name: str = "dccuchile/bert-base-spanish-wwm-cased"

    # Longitud máxima de tokens. Textos más largos se truncarán a este límite.
    max_length: int = 256

    # Parámetros estándar de optimización.
    learning_rate: float = 2e-5
    weight_decay: float = 0.01

    # Batch size para entrenamiento y evaluación.
    # Si aparece error de memoria, reduce train_batch_size y/o eval_batch_size.
    train_batch_size: int = 16
    eval_batch_size: int = 32

    # Número de épocas y acumulación de gradiente.
    num_train_epochs: int = 3
    gradient_accumulation_steps: int = 1

    # Warmup ayuda a estabilizar el entrenamiento al principio.
    warmup_ratio: float = 0.1

    # Semilla para reproducibilidad.
    seed: int = 42


@dataclass
class DataConfig:
    """Parámetros relacionados con el dataset y sus particiones."""

    dataset_path: Path = Path("../Data/Bin/Data_Bin_Classifier.csv")
    output_dir: Path = Path("./artifacts")

    # Proporciones globales: test_size=0.2 deja 20% para test.
    # val_size=0.1 deja 10% del total para validación.
    test_size: float = 0.2
    val_size: float = 0.1

    # Semilla usada al crear splits estratificados.
    random_state: int = 42


def ensure_output_dirs(base_dir: Path) -> dict:
    """Crea y devuelve las carpetas de salida del experimento.

    Devuelve un diccionario con rutas para:
    - base: carpeta raíz de artefactos,
    - model: modelo/tokenizer entrenados,
    - metrics: JSON/CSV de métricas y predicciones,
    - explanations: archivos HTML/JSON de explicabilidad.
    """

    base_dir.mkdir(parents=True, exist_ok=True)
    dirs = {
        "base": base_dir,
        "model": base_dir / "model",
        "metrics": base_dir / "metrics",
        "explanations": base_dir / "explanations",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


## 4. Utilidades de carga, limpieza y split del dataset

El dataset final debe quedar con dos columnas: `text` y `label`, donde `label=1` representa FAKE y `label=0` representa REAL/no fake.

In [ ]:
import re
import unicodedata
from html import unescape
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Mapeo de etiquetas textuales a formato binario.
# Convención del proyecto: 1 = FAKE, 0 = REAL / no fake.
LABEL_MAP = {
    "fake": 1,
    "false": 1,
    "1": 1,
    "true": 0,
    "real": 0,
    "no fake": 0,
    "0": 0,
}


def minimal_beto_clean(text: str) -> str:
    """Aplica una limpieza mínima pensada para modelos tipo BERT/BETO.

    No eliminamos signos de puntuación ni acentos porque pueden aportar señal
    lingüística útil y porque BETO fue preentrenado con texto natural en español.
    La limpieza se limita a normalizar HTML, unicode y espacios raros.
    """

    # Convierte NaN/None en cadena vacía y asegura tipo str.
    text = "" if pd.isna(text) else str(text)

    # Convierte entidades HTML como &amp; o &quot; a caracteres normales.
    text = unescape(text)

    # Normaliza unicode para evitar variantes visualmente equivalentes.
    text = unicodedata.normalize("NFKC", text)

    # Sustituye espacios no separables o invisibles por espacios normales.
    text = text.replace("\u00a0", " ").replace("\u200b", " ")

    # Colapsa múltiples espacios/saltos de línea en un único espacio.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _pick_text_column(df: pd.DataFrame) -> str:
    """Selecciona la columna de texto más adecuada del CSV.

    El dataset puede venir con distintos nombres de columna. Esta función intenta
    primero columnas comunes y, si existen `Title` y `Content`, crea `full_text`
    concatenando título y cuerpo de la noticia.
    """

    preferred = ["text", "text_beto", "full_text", "Content"]
    for col in preferred:
        if col in df.columns:
            return col

    # Caso frecuente en datasets de noticias: título + contenido.
    if "Title" in df.columns and "Content" in df.columns:
        df["full_text"] = (
            (
                df["Title"].fillna("").astype(str).str.strip()
                + " "
                + df["Content"].fillna("").astype(str).str.strip()
            )
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        return "full_text"

    raise KeyError(
        "No se encontró una columna de texto usable. Esperadas: text, text_beto, full_text o Content."
    )


def _normalize_labels(df: pd.DataFrame) -> pd.Series:
    """Normaliza etiquetas a enteros 0/1.

    Prioriza una columna `label` ya binaria. Si no existe, intenta mapear la
    columna `State` usando LABEL_MAP.
    """

    # Si ya existe label y todos sus valores son 0/1, la usamos directamente.
    if "label" in df.columns:
        label_series = pd.to_numeric(df["label"], errors="coerce")
        if label_series.notna().all():
            label_series = label_series.astype(int)
            if set(label_series.unique()).issubset({0, 1}):
                return label_series

    # Si no hay label válida, esperamos una columna textual `State`.
    if "State" not in df.columns:
        raise KeyError(
            "No existe la columna 'State' y tampoco una columna 'label' válida."
        )

    state = df["State"].astype(str).str.strip().str.lower()
    labels = state.map(LABEL_MAP)

    # Mostrar las etiquetas desconocidas ayuda a corregir el CSV o ampliar LABEL_MAP.
    if labels.isna().any():
        unknown = df.loc[labels.isna(), "State"].value_counts()
        raise ValueError(f"Etiquetas State no mapeadas: {unknown.to_dict()}")

    return labels.astype(int)


def load_binary_dataset(dataset_path: Path) -> pd.DataFrame:
    """Carga el CSV y devuelve un DataFrame limpio con columnas `text` y `label`."""

    dataset_path = Path(dataset_path)
    if not dataset_path.exists():
        raise FileNotFoundError(f"No se encontró dataset en: {dataset_path}")

    df = pd.read_csv(dataset_path)
    text_col = _pick_text_column(df)

    df = df.copy()
    df["text"] = df[text_col].map(minimal_beto_clean)

    # Quitamos textos vacíos porque no aportan señal y pueden romper explicadores.
    df = df[df["text"].str.len() > 0].copy()
    df["label"] = _normalize_labels(df)

    # Evita que la misma noticia aparezca en train y test tras el split.
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return df[["text", "label"]]


def stratified_splits(
    df: pd.DataFrame,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42,
):
    """Divide el dataset en train, validation y test manteniendo proporción de clases.

    La estratificación es importante en clasificación fake/no fake para evitar
    que un split quede desbalanceado si el dataset no tiene clases 50/50.
    """

    # Primero se separa test del resto.
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=random_state,
    )

    # val_size está expresado respecto al total original, por eso se convierte
    # a proporción relativa del bloque train_val.
    val_relative_size = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_relative_size,
        stratify=train_val_df["label"],
        random_state=random_state,
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


## 5. Vista rápida del dataset

Ejecuta esta celda para validar columnas, distribución de clases y tamaños de train/validation/test antes de entrenar.

In [ ]:
if DATASET_PATH.exists():
    df = load_binary_dataset(DATASET_PATH)
    train_df, val_df, test_df = stratified_splits(df)
    print(f"Total: {len(df)}")
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    display(df.head())
    display(df["label"].value_counts().rename(index={0: "REAL", 1: "FAKE"}).to_frame("n"))
else:
    print(f"No se encontró el dataset en: {DATASET_PATH.resolve()}")
    print("Ajusta DATASET_PATH en la celda de configuración antes de continuar.")

## 6. Entrenamiento de BETO

Esta sección incluye las funciones originales de fine-tuning, adaptadas para ejecutarse directamente en el notebook.

In [ ]:
import json
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# En el notebook no usamos imports relativos porque las funciones ya están definidas
# en celdas anteriores. En los scripts originales esto venía de .config y .data_utils.


def _build_hf_dataset(df: pd.DataFrame, tokenizer, max_length: int):
    """Convierte un DataFrame de pandas en un Dataset tokenizado de Hugging Face.

    Entrada esperada:
    - df con columnas `text` y `label`.
    - tokenizer compatible con BETO.

    Salida:
    - Dataset con `input_ids`, `attention_mask` y `label`.
    """

    # preserve_index=False evita añadir una columna extra con el índice de pandas.
    ds = Dataset.from_pandas(df, preserve_index=False)

    def tokenize_batch(batch):
        """Tokeniza por lotes para que el proceso sea más eficiente."""
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
        )

    ds = ds.map(tokenize_batch, batched=True)
    return ds


def _compute_metrics(eval_pred):
    """Calcula métricas de clasificación durante la evaluación del Trainer."""

    logits, labels = eval_pred

    # El modelo devuelve logits por clase. La clase predicha es la de logit máximo.
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
        "precision": precision_score(labels, preds, average="binary", zero_division=0),
        "recall": recall_score(labels, preds, average="binary", zero_division=0),
    }


def train_beto(
    dataset_path: Path,
    output_dir: Path,
    train_cfg: TrainConfig | None = None,
    data_cfg: DataConfig | None = None,
):
    """Entrena BETO para clasificación binaria REAL/FAKE.

    Pasos principales:
    1. cargar y limpiar dataset,
    2. crear splits estratificados,
    3. cargar tokenizer y modelo BETO,
    4. tokenizar train/validation,
    5. entrenar con Hugging Face Trainer,
    6. guardar modelo, tokenizer, splits y métricas.
    """

    # Si no se pasan configuraciones, usamos los valores por defecto.
    train_cfg = train_cfg or TrainConfig()
    data_cfg = data_cfg or DataConfig(dataset_path=dataset_path, output_dir=output_dir)

    dirs = ensure_output_dirs(Path(output_dir))
    set_seed(train_cfg.seed)

    print(f"[DATA] Cargando dataset desde {dataset_path}")
    df = load_binary_dataset(dataset_path)
    train_df, val_df, test_df = stratified_splits(
        df,
        test_size=data_cfg.test_size,
        val_size=data_cfg.val_size,
        random_state=data_cfg.random_state,
    )

    print(f"[DATA] Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")

    # Cargamos BETO y añadimos una cabeza de clasificación con 2 salidas.
    tokenizer = AutoTokenizer.from_pretrained(train_cfg.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        train_cfg.model_name,
        num_labels=2,
        id2label={0: "REAL", 1: "FAKE"},
        label2id={"REAL": 0, "FAKE": 1},
        use_safetensors=True,
    )

    # Tokenización de los splits usados por el Trainer.
    train_ds = _build_hf_dataset(train_df, tokenizer, train_cfg.max_length)
    val_ds = _build_hf_dataset(val_df, tokenizer, train_cfg.max_length)

    # Hugging Face Trainer solo necesita las columnas del modelo y la etiqueta.
    train_ds = train_ds.remove_columns(
        [
            c
            for c in train_ds.column_names
            if c not in {"input_ids", "attention_mask", "label"}
        ]
    )
    val_ds = val_ds.remove_columns(
        [
            c
            for c in val_ds.column_names
            if c not in {"input_ids", "attention_mask", "label"}
        ]
    )

    # El collator crea batches con padding dinámico al máximo largo de cada batch.
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_kwargs = {
        "output_dir": str(dirs["model"]),
        "learning_rate": train_cfg.learning_rate,
        "weight_decay": train_cfg.weight_decay,
        "per_device_train_batch_size": train_cfg.train_batch_size,
        "per_device_eval_batch_size": train_cfg.eval_batch_size,
        "num_train_epochs": train_cfg.num_train_epochs,
        "gradient_accumulation_steps": train_cfg.gradient_accumulation_steps,
        "warmup_ratio": train_cfg.warmup_ratio,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "report_to": "none",
        # fp16 se activa solo si hay GPU CUDA disponible.
        "fp16": torch.cuda.is_available(),
        "seed": train_cfg.seed,
    }

    # Compatibilidad entre versiones de transformers:
    # algunas usan evaluation_strategy y otras eval_strategy.
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    if "evaluation_strategy" in ta_params:
        training_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in ta_params:
        training_kwargs["eval_strategy"] = "epoch"

    training_args = TrainingArguments(**training_kwargs)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "eval_dataset": val_ds,
        "data_collator": data_collator,
        "compute_metrics": _compute_metrics,
    }

    # Compatibilidad entre versiones del Trainer: tokenizer vs processing_class.
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    print("[TRAIN] Iniciando fine-tuning de BETO...")
    trainer.train()

    # Métricas sobre validación usando el mejor checkpoint cargado al final.
    eval_metrics = trainer.evaluate()
    print("[EVAL] Métricas de validación:", eval_metrics)

    # Guardamos modelo/tokenizer para evaluación e inferencia posteriores.
    trainer.save_model(str(dirs["model"]))
    tokenizer.save_pretrained(str(dirs["model"]))

    # Guardar splits permite reproducir evaluación y revisar ejemplos concretos.
    train_df.to_csv(dirs["base"] / "train_split.csv", index=False)
    val_df.to_csv(dirs["base"] / "val_split.csv", index=False)
    test_df.to_csv(dirs["base"] / "test_split.csv", index=False)

    with open(dirs["metrics"] / "val_metrics.json", "w", encoding="utf-8") as f:
        json.dump(eval_metrics, f, indent=2, ensure_ascii=False)

    print(f"[OK] Modelo y artefactos guardados en {dirs['base']}")
    return {
        "model_dir": str(dirs["model"]),
        "splits": {
            "train": str(dirs["base"] / "train_split.csv"),
            "val": str(dirs["base"] / "val_split.csv"),
            "test": str(dirs["base"] / "test_split.csv"),
        },
        "metrics": str(dirs["metrics"] / "val_metrics.json"),
    }


### Ejecutar entrenamiento

Cambia `RUN_TRAIN = True` en la configuración para entrenar. El modelo y los splits se guardarán en `OUTPUT_DIR`.

In [ ]:
if RUN_TRAIN:
    train_cfg = TrainConfig(
        model_name="dccuchile/bert-base-spanish-wwm-cased",
        max_length=256,
        learning_rate=2e-5,
        weight_decay=0.01,
        train_batch_size=16,
        eval_batch_size=32,
        num_train_epochs=3,
        gradient_accumulation_steps=1,
        warmup_ratio=0.1,
        seed=42,
    )
    data_cfg = DataConfig(
        dataset_path=DATASET_PATH,
        output_dir=OUTPUT_DIR,
        test_size=0.2,
        val_size=0.1,
        random_state=42,
    )
    train_result = train_beto(DATASET_PATH, OUTPUT_DIR, train_cfg=train_cfg, data_cfg=data_cfg)
    train_result
else:
    print("Entrenamiento omitido. Cambia RUN_TRAIN = True para ejecutarlo.")

## 7. Evaluación en test

La evaluación carga el modelo entrenado desde `MODEL_DIR`, calcula métricas sobre el split de test y guarda artefactos en `OUTPUT_DIR/metrics`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# En el notebook estas funciones ya están definidas en celdas anteriores.


def _score_map_from_pipeline_sample(sample):
    """Convierte la salida del pipeline a un diccionario estable de probabilidades.

    Según la versión de transformers, el pipeline puede devolver:
    - una lista de diccionarios con todas las clases,
    - o un único diccionario con la clase más probable.

    Esta función normaliza ambas formas a algo como:
    {"REAL": 0.12, "FAKE": 0.88}
    """

    if isinstance(sample, list):
        return {item["label"].upper(): float(item["score"]) for item in sample}

    if isinstance(sample, dict) and "label" in sample and "score" in sample:
        label = str(sample["label"]).upper()
        score = float(sample["score"])
        if label in {"FAKE", "LABEL_1"}:
            return {"FAKE": score, "REAL": 1.0 - score}
        return {"REAL": score, "FAKE": 1.0 - score}

    return {}


def evaluate_model(model_dir: Path, dataset_path: Path, output_dir: Path):
    """Evalúa el modelo entrenado sobre el split de test.

    Genera dos salidas principales:
    - `test_metrics.json`: métricas agregadas como precision/recall/F1 y ROC-AUC.
    - `test_predictions.csv`: predicción y probabilidad de FAKE por noticia.
    """

    model_dir = Path(model_dir)
    output_dir = Path(output_dir)
    dirs = ensure_output_dirs(output_dir)

    # Reproduce el mismo split estratificado usado durante entrenamiento.
    df = load_binary_dataset(dataset_path)
    _, _, test_df = stratified_splits(df)

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    clf = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        top_k=None,
    )

    probs_fake = []
    preds = []

    for text in test_df["text"].tolist():
        # La truncación debe coincidir con el max_length usado durante entrenamiento.
        scores = clf(text, truncation=True, max_length=256)
        sample = scores[0] if isinstance(scores, list) and len(scores) == 1 else scores
        score_map = _score_map_from_pipeline_sample(sample)

        # Umbral simple: p(FAKE) >= 0.5 implica clase FAKE.
        p_fake = score_map.get("FAKE", score_map.get("LABEL_1", 0.0))
        probs_fake.append(p_fake)
        preds.append(1 if p_fake >= 0.5 else 0)

    y_true = test_df["label"].values
    y_pred = np.array(preds)

    report = classification_report(
        y_true,
        y_pred,
        target_names=["REAL", "FAKE"],
        output_dict=True,
        zero_division=0,
    )

    # ROC-AUC necesita que haya al menos dos clases presentes en y_true.
    try:
        auc = roc_auc_score(y_true, np.array(probs_fake))
    except ValueError:
        auc = None

    metrics = {
        "classification_report": report,
        "roc_auc": auc,
        "n_test": int(len(test_df)),
    }

    with open(dirs["metrics"] / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    pd.DataFrame(
        {
            "text": test_df["text"],
            "y_true": y_true,
            "y_pred": y_pred,
            "p_fake": probs_fake,
        }
    ).to_csv(dirs["metrics"] / "test_predictions.csv", index=False)

    print("[OK] Evaluación completada")
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    return metrics


### Ejecutar evaluación

In [ ]:
if RUN_EVALUATE:
    metrics = evaluate_model(model_dir=MODEL_DIR, dataset_path=DATASET_PATH, output_dir=OUTPUT_DIR)
    metrics
else:
    print("Evaluación omitida. Cambia RUN_EVALUATE = True para ejecutarla.")

### Ver métricas guardadas

In [ ]:
metrics_path = OUTPUT_DIR / "metrics" / "test_metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        saved_metrics = json.load(f)
    print(json.dumps(saved_metrics, indent=2, ensure_ascii=False))
else:
    print(f"Aún no existe: {metrics_path}")

## 8. Explicabilidad: LIME, SHAP, Integrated Gradients y faithfulness

Esta sección integra la clase `BetoExplainer` y el runner de explicaciones. Genera archivos HTML/JSON y un índice CSV con los resultados.

In [ ]:
import json
from pathlib import Path

import numpy as np
import shap
import torch
from captum.attr import LayerIntegratedGradients
from lime.lime_text import LimeTextExplainer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline


class BetoExplainer:
    """Agrupa métodos de explicabilidad para un modelo BETO ya entrenado.

    Incluye cuatro aproximaciones complementarias:
    - LIME: explica palabras influyentes perturbando el texto.
    - SHAP: estima contribuciones token/palabra con valores Shapley aproximados.
    - Integrated Gradients: atribución basada en gradientes sobre embeddings.
    - Faithfulness: mide cómo cambian las probabilidades al quitar/dejar palabras importantes.
    """

    def __init__(self, model_dir: Path):
        """Carga modelo, tokenizer y pipeline de clasificación."""

        self.model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()

        # Usa GPU si está disponible; si no, CPU.
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

        self.labels = ["REAL", "FAKE"]
        self.clf_pipeline = pipeline(
            "text-classification",
            model=self.model,
            tokenizer=self.tokenizer,
            top_k=None,
            device=0 if self.device.type == "cuda" else -1,
        )

    @staticmethod
    def _score_map_from_pipeline_sample(sample):
        """Normaliza la salida del pipeline a probabilidades por clase."""

        # API nueva: lista de diccionarios, uno por clase.
        if isinstance(sample, list):
            return {item["label"].upper(): float(item["score"]) for item in sample}

        # Fallback: salida con una sola etiqueta y score.
        if isinstance(sample, dict) and "label" in sample and "score" in sample:
            label = str(sample["label"]).upper()
            score = float(sample["score"])
            if label in {"FAKE", "LABEL_1"}:
                return {"FAKE": score, "REAL": 1.0 - score}
            return {"REAL": score, "FAKE": 1.0 - score}

        return {}

    def predict_proba(self, texts):
        """Devuelve matriz de probabilidades [[p_REAL, p_FAKE], ...].

        Este formato es el que esperan explicadores como LIME.
        """

        outputs = self.clf_pipeline(list(texts), truncation=True, max_length=256)
        probs = []
        for sample in outputs:
            score_map = self._score_map_from_pipeline_sample(sample)
            p_real = score_map.get("REAL", score_map.get("LABEL_0", 0.0))
            p_fake = score_map.get("FAKE", score_map.get("LABEL_1", 0.0))
            probs.append([p_real, p_fake])
        return np.array(probs)

    def _predict_with_details(self, text: str):
        """Predice un texto y devuelve probabilidades, clase y probabilidad ganadora."""

        probs = self.predict_proba([text])[0]
        pred_class = int(np.argmax(probs))
        pred_prob = float(probs[pred_class])
        return probs, pred_class, pred_prob

    @staticmethod
    def _normalize_word(word: str) -> str:
        """Normaliza una palabra para comparaciones simples en explicabilidad."""

        return "".join(
            ch
            for ch in word.lower()
            if ch.isalnum() or ch in {"_", "-", "á", "é", "í", "ó", "ú", "ñ", "ü"}
        )

    def _word_occlusion_importance(self, text: str, target_class: int):
        """Calcula importancia por oclusión palabra a palabra.

        Para cada palabra, la elimina del texto y mide cuánto baja la probabilidad
        de la clase objetivo. Si baja mucho, esa palabra era importante para esa clase.
        """

        words = text.split()
        if not words:
            return [], []

        base_prob = float(self.predict_proba([text])[0][target_class])
        ranked = []

        for idx in range(len(words)):
            perturbed_words = words[:idx] + words[idx + 1 :]
            perturbed_text = " ".join(perturbed_words).strip()

            if not perturbed_text:
                perturbed_prob = 0.0
            else:
                perturbed_prob = float(
                    self.predict_proba([perturbed_text])[0][target_class]
                )

            importance = base_prob - perturbed_prob
            ranked.append(
                {
                    "index": idx,
                    "word": words[idx],
                    "importance": float(importance),
                    "prob_without_word": float(perturbed_prob),
                }
            )

        ranked = sorted(ranked, key=lambda x: x["importance"], reverse=True)
        return words, ranked

    @staticmethod
    def _build_text_from_indices(words, selected_indices, keep_selected=True):
        """Reconstruye texto manteniendo o eliminando ciertos índices de palabras."""

        selected = set(selected_indices)
        if keep_selected:
            out = [w for i, w in enumerate(words) if i in selected]
        else:
            out = [w for i, w in enumerate(words) if i not in selected]
        return " ".join(out).strip()

    @staticmethod
    def _auc(points):
        """Calcula área bajo una curva definida por puntos {fraction, prob}."""

        if len(points) < 2:
            return None
        points = sorted(points, key=lambda x: x["fraction"])
        x = np.array([p["fraction"] for p in points], dtype=float)
        y = np.array([p["prob"] for p in points], dtype=float)
        if hasattr(np, "trapezoid"):
            return float(np.trapezoid(y, x))
        return float(np.trapz(y, x))

    def explain_lime(self, text: str, output_path: Path, num_features: int = 12):
        """Genera explicación LIME y la guarda como HTML.

        LIME perturba el texto, observa cambios en la predicción y ajusta un modelo
        local interpretable para estimar qué palabras influyen más.
        """

        explainer = LimeTextExplainer(class_names=self.labels)
        exp = explainer.explain_instance(
            text,
            classifier_fn=self.predict_proba,
            labels=[0, 1],
            num_features=num_features,
        )

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(exp.as_html())

        return {
            "path": str(output_path),
            "top_real": exp.as_list(label=0),
            "top_fake": exp.as_list(label=1),
        }

    def explain_shap(self, texts, output_path: Path, max_samples: int = 8):
        """Genera resumen SHAP para una lista de textos y lo guarda en JSON.

        Para evitar errores con textos largos, antes de llamar a SHAP se truncan
        los textos con el mismo tokenizer y max_length usado por el modelo.
        """

        texts = list(texts)[:max_samples]

        truncated_texts = []
        for text in texts:
            encoded = self.tokenizer(
                text,
                truncation=True,
                max_length=256,
                return_tensors=None,
            )
            truncated_text = self.tokenizer.decode(
                encoded["input_ids"],
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )
            truncated_texts.append(truncated_text)

        explainer = shap.Explainer(self.clf_pipeline)
        shap_values = explainer(truncated_texts)

        records = []
        for i, text in enumerate(truncated_texts):
            sample_values = shap_values.values[i]
            sample_data = shap_values.data[i]
            probs = self.predict_proba([text])[0]
            pred_class = int(np.argmax(probs))

            token_scores = []
            for token, scores in zip(sample_data, sample_values):
                token_scores.append(
                    {
                        "token": str(token),
                        "score": float(scores[pred_class]),
                    }
                )

            # Ordenamos por magnitud absoluta para quedarnos con tokens más influyentes.
            token_scores = sorted(
                token_scores, key=lambda x: abs(x["score"]), reverse=True
            )

            records.append(
                {
                    "text": text,
                    "predicted_class": self.labels[pred_class],
                    "top_tokens": token_scores[:20],
                }
            )

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

        return {"path": str(output_path), "samples": len(records)}

    def explain_integrated_gradients(
        self, text: str, output_path: Path, target: int | None = None
    ):
        """Calcula Integrated Gradients sobre los embeddings de BETO.

        Integrated Gradients atribuye importancia a tokens comparando el input real
        con una trayectoria desde una línea base implícita hasta el input.
        """

        encoded = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
        )
        input_ids = encoded["input_ids"].to(self.device)
        attention_mask = encoded["attention_mask"].to(self.device)

        with torch.no_grad():
            logits = self.model(
                input_ids=input_ids, attention_mask=attention_mask
            ).logits
            pred = int(torch.argmax(logits, dim=1).item())

        # Si no se especifica target, explicamos la clase predicha por el modelo.
        target = pred if target is None else int(target)

        def forward_func(ids, mask):
            """Forward requerido por Captum: devuelve logits del modelo."""
            out = self.model(input_ids=ids, attention_mask=mask).logits
            return out

        lig = LayerIntegratedGradients(forward_func, self.model.get_input_embeddings())
        attributions, delta = lig.attribute(
            inputs=input_ids,
            additional_forward_args=(attention_mask,),
            target=target,
            n_steps=32,
            return_convergence_delta=True,
        )

        # Sumamos atribuciones en la dimensión del embedding para obtener score por token.
        token_attr = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).tolist())

        token_scores = [
            {"token": tok, "score": float(score)}
            for tok, score in zip(tokens, token_attr)
            if tok not in {"[CLS]", "[SEP]", "[PAD]"}
        ]
        token_scores = sorted(token_scores, key=lambda x: abs(x["score"]), reverse=True)

        output = {
            "predicted_class": self.labels[pred],
            "target_class": self.labels[target],
            "convergence_delta": float(torch.mean(torch.abs(delta)).item()),
            "top_tokens": token_scores[:25],
        }

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

        return {"path": str(output_path), "predicted_class": self.labels[pred]}

    def explain_faithfulness(
        self,
        text: str,
        output_path: Path,
        fractions=(0.1, 0.2, 0.3, 0.4, 0.5),
    ):
        """Evalúa faithfulness de las palabras importantes por oclusión.

        Métricas principales:
        - comprehensiveness: cuánto cae la probabilidad al quitar palabras top.
        - sufficiency: cuánto se conserva la predicción usando solo palabras top.
        - deletion_auc / insertion_auc: áreas bajo curvas de borrado/inserción.
        """

        probs, pred_class, base_prob = self._predict_with_details(text)
        words, ranked = self._word_occlusion_importance(text, target_class=pred_class)

        if not words:
            output = {
                "predicted_class": self.labels[pred_class],
                "base_prob": base_prob,
                "error": "Texto vacío tras tokenización por espacios.",
            }
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(output, f, indent=2, ensure_ascii=False)
            return {
                "path": str(output_path),
                "predicted_class": self.labels[pred_class],
                "mean_comprehensiveness": float("nan"),
                "mean_sufficiency": float("nan"),
                "deletion_auc": float("nan"),
                "insertion_auc": float("nan"),
                "n_words": 0,
                "base_pred_prob": float(base_prob),
            }

        top_indices = [item["index"] for item in ranked]
        n_words = len(words)

        comp_points = []
        suff_points = []
        deletion_curve = []
        insertion_curve = []

        for frac in fractions:
            k = max(1, int(round(frac * n_words)))
            selected = top_indices[:k]

            # Comprehensiveness: quitar las palabras más importantes.
            comp_text = self._build_text_from_indices(
                words, selected, keep_selected=False
            )
            # Sufficiency: conservar solo las palabras más importantes.
            suff_text = self._build_text_from_indices(
                words, selected, keep_selected=True
            )

            comp_prob = (
                0.0
                if not comp_text
                else float(self.predict_proba([comp_text])[0][pred_class])
            )
            suff_prob = (
                0.0
                if not suff_text
                else float(self.predict_proba([suff_text])[0][pred_class])
            )

            comprehensiveness = base_prob - comp_prob
            sufficiency = base_prob - suff_prob

            point = {
                "fraction": float(frac),
                "k": int(k),
                "prob": float(comp_prob),
                "delta": float(comprehensiveness),
            }
            comp_points.append(point)
            deletion_curve.append(point)

            point = {
                "fraction": float(frac),
                "k": int(k),
                "prob": float(suff_prob),
                "delta": float(sufficiency),
            }
            suff_points.append(point)
            insertion_curve.append(point)

        output = {
            "predicted_class": self.labels[pred_class],
            "base_probs": {
                "REAL": float(probs[0]),
                "FAKE": float(probs[1]),
            },
            "base_pred_prob": float(base_prob),
            "n_words": int(n_words),
            "top_words": ranked[:25],
            "comprehensiveness": {
                "points": comp_points,
                "mean_delta": float(np.mean([p["delta"] for p in comp_points])),
            },
            "sufficiency": {
                "points": suff_points,
                "mean_delta": float(np.mean([p["delta"] for p in suff_points])),
            },
            "deletion_curve": {
                "points": deletion_curve,
                "auc": self._auc(deletion_curve),
            },
            "insertion_curve": {
                "points": insertion_curve,
                "auc": self._auc(insertion_curve),
            },
            "notes": {
                "comprehensiveness": "Mayor delta indica mayor dependencia del modelo en palabras top.",
                "sufficiency": "Delta cercano a 0 indica que las palabras top retienen suficiente evidencia.",
            },
        }

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

        return {
            "path": str(output_path),
            "predicted_class": self.labels[pred_class],
            "mean_comprehensiveness": output["comprehensiveness"]["mean_delta"],
            "mean_sufficiency": output["sufficiency"]["mean_delta"],
            "deletion_auc": output["deletion_curve"]["auc"],
            "insertion_auc": output["insertion_curve"]["auc"],
            "n_words": int(n_words),
            "base_pred_prob": float(base_prob),
        }


In [ ]:
import json
from pathlib import Path

import pandas as pd

# En el notebook BetoExplainer y las utilidades de datos ya están definidas arriba.


def _series_stats(series: pd.Series):
    """Resume una serie numérica con estadísticos básicos.

    Se usa para resumir métricas de faithfulness globales, tanto en conjunto
    completo como agrupadas por clase predicha.
    """

    clean = series.dropna()
    if clean.empty:
        return {
            "count": 0,
            "mean": None,
            "std": None,
            "median": None,
            "p25": None,
            "p75": None,
            "min": None,
            "max": None,
        }

    return {
        "count": int(clean.shape[0]),
        "mean": float(clean.mean()),
        "std": float(clean.std(ddof=0)),
        "median": float(clean.median()),
        "p25": float(clean.quantile(0.25)),
        "p75": float(clean.quantile(0.75)),
        "min": float(clean.min()),
        "max": float(clean.max()),
    }


def _build_faithfulness_summary(df: pd.DataFrame):
    """Construye un resumen global de métricas de faithfulness.

    Además del resumen general, agrupa por clase predicha para revisar si las
    explicaciones se comportan diferente cuando el modelo predice REAL o FAKE.
    """

    metrics = [
        "mean_comprehensiveness",
        "mean_sufficiency",
        "deletion_auc",
        "insertion_auc",
        "base_pred_prob",
        "n_words",
    ]

    overall = {metric: _series_stats(df[metric]) for metric in metrics}

    by_class = {}
    for pred_class, group in df.groupby("predicted_class"):
        by_class[str(pred_class)] = {
            metric: _series_stats(group[metric]) for metric in metrics
        }

    return {
        "n_samples": int(df.shape[0]),
        "overall": overall,
        "by_predicted_class": by_class,
    }


def run_explanations(
    model_dir: Path,
    output_dir: Path,
    dataset_path: Path | None = None,
    text: str | None = None,
    n_samples: int = 3,
    faithfulness_scope: str = "sample",
    faithfulness_max_samples: int | None = None,
    random_state: int = 42,
):
    """Genera explicaciones y métricas de faithfulness.

    Modos de uso:
    - Si `text` no es None: explica únicamente ese texto.
    - Si `text` es None: toma ejemplos del split de test del dataset.
    - Si faithfulness_scope="full_test": calcula faithfulness para todo test
      o para un subconjunto limitado por `faithfulness_max_samples`.
    """

    dirs = ensure_output_dirs(Path(output_dir))
    explainer = BetoExplainer(model_dir)

    if text:
        # Modo explicación de un texto manual.
        sample_texts = [text]
    else:
        if dataset_path is None:
            raise ValueError(
                "Si no se pasa texto directo, dataset_path es obligatorio."
            )
        df = load_binary_dataset(dataset_path)
        _, _, test_df = stratified_splits(df)

        if faithfulness_scope == "full_test":
            work_df = test_df.copy()
            if faithfulness_max_samples is not None and faithfulness_max_samples > 0:
                work_df = work_df.sample(
                    n=min(int(faithfulness_max_samples), len(work_df)),
                    random_state=random_state,
                )
            sample_texts = work_df["text"].tolist()
        else:
            sample_texts = test_df["text"].head(n_samples).tolist()

    rows = []
    for i, sample in enumerate(sample_texts, start=1):
        # Cada muestra produce varios archivos: LIME HTML, IG JSON y faithfulness JSON.
        lime_path = dirs["explanations"] / f"lime_sample_{i}.html"
        ig_path = dirs["explanations"] / f"ig_sample_{i}.json"
        faithfulness_path = dirs["explanations"] / f"faithfulness_sample_{i}.json"

        lime_result = explainer.explain_lime(sample, lime_path)
        ig_result = explainer.explain_integrated_gradients(sample, ig_path)
        faith_result = explainer.explain_faithfulness(sample, faithfulness_path)

        rows.append(
            {
                "sample_id": i,
                "text": sample,
                "lime_html": lime_result["path"],
                "ig_json": ig_result["path"],
                "faithfulness_json": faith_result["path"],
                "predicted_class": ig_result["predicted_class"],
                "mean_comprehensiveness": faith_result["mean_comprehensiveness"],
                "mean_sufficiency": faith_result["mean_sufficiency"],
                "deletion_auc": faith_result["deletion_auc"],
                "insertion_auc": faith_result["insertion_auc"],
                "n_words": faith_result["n_words"],
                "base_pred_prob": faith_result["base_pred_prob"],
            }
        )

    # SHAP se calcula sobre las mismas muestras. Se limita a 50 para evitar coste excesivo.
    shap_path = dirs["explanations"] / "shap_summary.json"
    explainer.explain_shap(
        sample_texts, shap_path, max_samples=min(50, len(sample_texts))
    )

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(dirs["explanations"] / "explanations_index.csv", index=False)

    faithfulness_summary = _build_faithfulness_summary(summary_df)
    faithfulness_summary["scope"] = faithfulness_scope
    faithfulness_summary["max_samples"] = faithfulness_max_samples

    faithfulness_summary_path = (
        dirs["explanations"] / "faithfulness_global_summary.json"
    )
    with open(faithfulness_summary_path, "w", encoding="utf-8") as f:
        json.dump(faithfulness_summary, f, indent=2, ensure_ascii=False)

    print("[OK] Explicaciones generadas")
    print(f"- Carpeta: {dirs['explanations']}")
    return {
        "index": str(dirs["explanations"] / "explanations_index.csv"),
        "shap": str(shap_path),
        "faithfulness_global": str(faithfulness_summary_path),
    }


### Ejecutar explicaciones

- Si `EXPLAIN_TEXT` contiene un texto, se explica ese texto.
- Si `EXPLAIN_TEXT = None`, se explican muestras del split de test.
- Para métricas globales de faithfulness, usa `FAITHFULNESS_SCOPE = "full_test"` y opcionalmente limita con `FAITHFULNESS_MAX_SAMPLES`.

In [ ]:
if RUN_EXPLAIN:
    explanation_result = run_explanations(
        model_dir=MODEL_DIR,
        output_dir=OUTPUT_DIR,
        dataset_path=DATASET_PATH,
        text=EXPLAIN_TEXT,
        n_samples=EXPLAIN_N_SAMPLES,
        faithfulness_scope=FAITHFULNESS_SCOPE,
        faithfulness_max_samples=FAITHFULNESS_MAX_SAMPLES,
        random_state=42,
    )
    explanation_result
else:
    print("Explicabilidad omitida. Cambia RUN_EXPLAIN = True para ejecutarla.")

### Revisar índice de explicaciones

In [ ]:
index_path = OUTPUT_DIR / "explanations" / "explanations_index.csv"
if index_path.exists():
    explanations_df = pd.read_csv(index_path)
    display(explanations_df)
else:
    print(f"Aún no existe: {index_path}")

## 9. Inferencia rápida con el modelo entrenado

Usa esta celda para probar un texto manualmente una vez que exista `MODEL_DIR`.

In [ ]:
def predict_news_label(text: str, model_dir: Path = MODEL_DIR):
    """Clasifica un texto nuevo con el modelo BETO entrenado.

    Devuelve la salida cruda del pipeline de Hugging Face, normalmente una lista
    con probabilidades para REAL y FAKE. Es útil para pruebas rápidas tras entrenar.
    """

    from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    clf = pipeline("text-classification", model=model, tokenizer=tokenizer, top_k=None)

    # Mantenemos truncation y max_length para que inferencia sea consistente con entrenamiento.
    scores = clf(text, truncation=True, max_length=256)
    return scores


# Escribe aquí una noticia o fragmento para probar la inferencia.
sample_text = ""

if sample_text and MODEL_DIR.exists():
    predict_news_label(sample_text)
elif not MODEL_DIR.exists():
    print(f"No se encontró modelo en: {MODEL_DIR}")
else:
    print("Escribe un texto en sample_text para probar inferencia.")


## 10. Salidas esperadas

Después de ejecutar el flujo completo, deberías tener:

- `artifacts/model/`: modelo BETO fine-tuned y tokenizer.
- `artifacts/train_split.csv`, `val_split.csv`, `test_split.csv`: divisiones estratificadas.
- `artifacts/metrics/val_metrics.json`: métricas de validación.
- `artifacts/metrics/test_metrics.json`: métricas de test.
- `artifacts/metrics/test_predictions.csv`: predicciones del test.
- `artifacts/explanations/`: explicaciones LIME/SHAP/IG, métricas de faithfulness e índice CSV.